# SWEPAM Instrument Response and Solar Wind Moments / SWEPAM 기기 응답과 태양풍 모멘트

**Paper**: McComas, D. J., Bame, S. J., Barker, P., Feldman, W. C., Phillips, J. L., Riley, P., and Griffee, J. W., "Solar Wind Electron Proton Alpha Monitor (SWEPAM) for the Advanced Composition Explorer", *Space Sci. Rev.* **86**, 563-612, 1998. DOI: 10.1023/A:1005040232597

## 노트북 목적 / Notebook purpose

이 노트북은 McComas et al. (1998)의 SWEPAM 기기 보정과 데이터 처리 핵심 개념을 합성 데이터로 재현한다. 네 부분으로 나뉜다: (1) SWEPAM-I 이온 ESA의 속도 공간 응답, (2) SWEPAM-E 전자 채널의 각도 응답, (3) 측정된 카운트로부터 VDF 모멘트 (밀도, 속도, 온도) 계산, (4) 고속 풍(HSS) vs 저속 풍 군집을 $n$-$T$ 평면에서 분리.

This notebook reproduces the key calibration and data-processing concepts of McComas et al. (1998) using synthetic data. It covers four parts: (1) SWEPAM-I ion ESA velocity-space response, (2) SWEPAM-E electron angular response, (3) computing VDF moments (density, velocity, temperature) from simulated counts, and (4) separating high-speed-stream (HSS) vs slow-wind populations in $n$-$T$ space.

**Kernel**: `study-with-ai` (conda env)

In [ ]:
"""Imports and global plotting style for the SWEPAM notebook."""

import numpy as np
import matplotlib.pyplot as plt
from scipy import constants
from scipy.optimize import curve_fit
from scipy.special import erf

rng = np.random.default_rng(seed=20260425)

plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 12
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

# Physical constants (SI)
M_P = constants.m_p           # proton mass [kg]
M_E = constants.m_e           # electron mass [kg]
K_B = constants.k             # Boltzmann constant [J/K]
EV = constants.e              # 1 eV in joules

print(f"m_p = {M_P:.3e} kg, m_e = {M_E:.3e} kg, k_B = {K_B:.3e} J/K")

## Part 1: SWEPAM-I Ion ESA Velocity-Space Response / SWEPAM-I 이온 ESA 속도 공간 응답

SWEPAM-I는 105° 곡판형 ESA로 한 픽셀당 좁은 $E/q$ 영역과 좁은 방위각 fan을 통과시킨다. Gosling et al. (1978)에 따라 응답 함수 $T(E, \theta, \phi)$는 거의 분리 가능하며, 각각 가우시안 (energy, azimuth)과 사다리꼴 (polar)로 모델링된다. 분석기 상수는 $E/q = K \cdot V_{\rm plate}$로 $K \approx 17.1$ (측정값).

SWEPAM-I uses a 105° spherical-section ESA that, per pixel, transmits a narrow $E/q$ band and a narrow azimuth fan. Following Gosling et al. (1978), the response $T(E, \theta, \phi)$ approximately separates into a Gaussian in energy and azimuth and a trapezoid in polar, with analyzer constant $K \approx 17.1$ (measured).

In [ ]:
def gaussian(x: np.ndarray, x0: float, fwhm: float) -> np.ndarray:
    """Normalized Gaussian (peak = 1) given a FWHM.

    Args:
        x: Input coordinate.
        x0: Peak location.
        fwhm: Full width at half maximum.

    Returns:
        Gaussian profile evaluated at ``x``.
    """
    sigma = fwhm / (2.0 * np.sqrt(2.0 * np.log(2.0)))
    return np.exp(-0.5 * ((x - x0) / sigma) ** 2)


def trapezoid(theta: np.ndarray, theta0: float, flat_width: float, ramp: float) -> np.ndarray:
    """Trapezoidal polar response with a flat top and linear ramp shoulders.

    Args:
        theta: Polar angle [deg].
        theta0: Center polar angle.
        flat_width: Width of the flat top [deg].
        ramp: Width of the linear ramp on each side [deg].

    Returns:
        Trapezoid profile in [0, 1].
    """
    dx = np.abs(theta - theta0)
    half_flat = flat_width / 2.0
    return np.where(
        dx <= half_flat,
        1.0,
        np.clip(1.0 - (dx - half_flat) / ramp, 0.0, 1.0),
    )


def swepam_i_response(E: np.ndarray, theta: np.ndarray, phi: np.ndarray,
                      E0: float, theta0: float, phi0: float,
                      dE_over_E: float = 0.05,
                      polar_flat: float = 5.0,
                      polar_ramp: float = 1.5,
                      azim_fwhm: float = 3.6) -> np.ndarray:
    """Separable SWEPAM-I response T(E, theta, phi).

    Args:
        E: Energy [eV].
        theta: Polar angle [deg].
        phi: Azimuth angle [deg].
        E0: Pixel-center energy [eV].
        theta0: Pixel-center polar [deg].
        phi0: Pixel-center azimuth [deg].
        dE_over_E: Energy FWHM divided by E0 (default 5%).
        polar_flat: Trapezoid flat-top width [deg].
        polar_ramp: Trapezoid ramp width [deg].
        azim_fwhm: Azimuth FWHM [deg].

    Returns:
        Element-wise response value in [0, 1].
    """
    return (gaussian(E, E0, dE_over_E * E0)
            * trapezoid(theta, theta0, polar_flat, polar_ramp)
            * gaussian(phi, phi0, azim_fwhm))


# Evaluate one-dimensional cuts of the response at a representative pixel
# (E0, theta0, phi0) = (1000 eV, 0 deg, 0 deg).
E_grid = np.linspace(800, 1200, 401)
theta_grid = np.linspace(-15, 15, 301)
phi_grid = np.linspace(-10, 10, 301)

E0, theta0, phi0 = 1000.0, 0.0, 0.0
TE = swepam_i_response(E_grid, theta0, phi0, E0, theta0, phi0)
Tth = swepam_i_response(E0, theta_grid, phi0, E0, theta0, phi0)
Tphi = swepam_i_response(E0, theta0, phi_grid, E0, theta0, phi0)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(E_grid, TE, color="C0")
axes[0].axvspan(E0 * 0.975, E0 * 1.025, color="C0", alpha=0.1, label="\u00b12.5%")
axes[0].set_xlabel("Energy [eV]")
axes[0].set_ylabel("Response")
axes[0].set_title("Energy: Gaussian, FWHM = 5%")
axes[0].legend()
axes[1].plot(theta_grid, Tth, color="C1")
axes[1].set_xlabel("Polar \u03b8 [deg]")
axes[1].set_title("Polar: trapezoid (flat \u22485\u00b0)")
axes[2].plot(phi_grid, Tphi, color="C2")
axes[2].set_xlabel("Azimuth \u03c6 [deg]")
axes[2].set_title("Azimuth: Gaussian, FWHM \u22483.6\u00b0")
for ax in axes:
    ax.set_ylim(-0.05, 1.1)
fig.suptitle("SWEPAM-I separable response function (Gosling et al. 1978)")
plt.tight_layout()
plt.show()

### 1.1 Analyzer-constant verification / 분석기 상수 검증

이상적 곡판 ESA에서 $E/q = K_{\rm ideal} V$, $K_{\rm ideal} = r_0 / (2 d)$. SWEPAM-I: $r_0 = 100$ mm, $d = 2.84$ mm $\to K_{\rm ideal} \approx 17.6$ vs 측정 $K \approx 17.1$ — fringing field 보정 후 $\sim 3\%$ 일치. SWEPAM-E의 큰 편차(~28%)는 120° 굴절각의 fringing field와 inner mesh가 plate 구간을 사실상 단축시키는 효과를 반영한다.

Ideal spherical-section ESA: $E/q = K_{\rm ideal} V$ with $K_{\rm ideal} = r_0/(2d)$. SWEPAM-I numbers give $K_{\rm ideal} \approx 17.6$ vs the calibrated 17.1 — agreement to ~3% after fringing-field correction. The larger SWEPAM-E deviation (~28%) reflects 120°-bend fringing fields and effective shortening of the bend by the inner mesh.

In [ ]:
def analyzer_constant(r0_mm: float, d_mm: float) -> float:
    """Ideal analyzer constant K = r0 / (2 d) for a spherical-section ESA.

    Args:
        r0_mm: Mean curvature radius [mm].
        d_mm: Plate-gap [mm].

    Returns:
        Ideal K (dimensionless).
    """
    return r0_mm / (2.0 * d_mm)


K_I_ideal = analyzer_constant(100.0, 2.84)
K_E_ideal = analyzer_constant(41.9, 3.5)
K_I_meas, K_E_meas = 17.1, 4.3

print(f"SWEPAM-I  ideal K = {K_I_ideal:.2f}, measured = {K_I_meas:.2f}, "
      f"deviation = {(K_I_meas - K_I_ideal) / K_I_ideal * 100:+.1f}%")
print(f"SWEPAM-E  ideal K = {K_E_ideal:.2f}, measured = {K_E_meas:.2f}, "
      f"deviation = {(K_E_meas - K_E_ideal) / K_E_ideal * 100:+.1f}%")

## Part 2: SWEPAM-E Electron Angular Response / SWEPAM-E 전자 각도 응답

SWEPAM-E는 120° 굴절각, 7개 CEM, 21° 폴라 간격을 가진다. Fig. 20c에 따르면 폴라 응답은 *왜곡 가우시안* (skewed Gaussian)이다. 7개 CEM은 폴라 $-63^\circ$, $-42^\circ$, $-21^\circ$, $0^\circ$, $+21^\circ$, $+42^\circ$, $+63^\circ$에 위치한다 (자전축에 수직 fan FOV).

SWEPAM-E uses a 120° bend, 7 CEMs at 21° polar spacing. Per Fig. 20c, the polar response is a *skewed Gaussian*. The 7 CEMs sit at polar angles $-63^\circ$, $-42^\circ$, $-21^\circ$, $0^\circ$, $+21^\circ$, $+42^\circ$, $+63^\circ$ (fan FOV perpendicular to the spin axis).

In [ ]:
def skewed_gaussian(x: np.ndarray, x0: float, fwhm: float, alpha: float) -> np.ndarray:
    """Skewed Gaussian normalized to peak = 1.

    Implementation: standard Gaussian multiplied by 2 * Phi(alpha * t),
    where t = (x - x0) / sigma and Phi is the normal CDF.

    Args:
        x: Coordinate.
        x0: Center used for the underlying Gaussian (mode shifts with alpha).
        fwhm: Approximate FWHM of the underlying Gaussian.
        alpha: Skewness parameter (0 = symmetric, > 0 = right-skewed).

    Returns:
        Profile rescaled so its maximum equals 1.
    """
    sigma = fwhm / (2.0 * np.sqrt(2.0 * np.log(2.0)))
    t = (x - x0) / sigma
    g = np.exp(-0.5 * t ** 2)
    s = 0.5 * (1.0 + erf(alpha * t / np.sqrt(2.0)))
    profile = g * 2.0 * s
    peak = profile.max()
    return profile / peak if peak > 0 else profile


swepam_e_polar_centers = np.array([-63.0, -42.0, -21.0, 0.0, 21.0, 42.0, 63.0])
polar_fine = np.linspace(-90.0, 90.0, 1801)

fig, ax = plt.subplots(figsize=(10, 5))
for i, theta_c in enumerate(swepam_e_polar_centers):
    profile = skewed_gaussian(polar_fine, theta_c, fwhm=21.0, alpha=0.6)
    ax.plot(polar_fine, profile, label=f"CEM {i + 1} (\u03b8\u2080={theta_c:+.0f}\u00b0)")
ax.set_xlabel("Polar angle \u03b8 [deg]")
ax.set_ylabel("Normalized response")
ax.set_title("SWEPAM-E polar response: 7 CEMs, skewed Gaussian (Fig. 20c, Fig. 21)")
ax.legend(ncol=4, fontsize=9)
ax.set_xlim(-90, 90)
ax.axhline(0.5, color="k", lw=0.5, ls="--", alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
# Fit a symmetric Gaussian to one CEM's skewed profile and quantify asymmetry.
profile_cem3 = skewed_gaussian(polar_fine, -21.0, fwhm=21.0, alpha=0.6)


def gauss_only(x, x0, fwhm, amp):
    return amp * gaussian(x, x0, fwhm)


popt, _ = curve_fit(
    gauss_only,
    polar_fine,
    profile_cem3,
    p0=[-21.0, 21.0, 1.0],
)
x0_fit, fwhm_fit, amp_fit = popt

above_half = profile_cem3 > 0.5
left = polar_fine[above_half & (polar_fine < x0_fit)]
right = polar_fine[above_half & (polar_fine > x0_fit)]
left_w = x0_fit - left.min() if left.size else np.nan
right_w = right.max() - x0_fit if right.size else np.nan

print(f"Symmetric-Gaussian fit: x0 = {x0_fit:.2f}, FWHM = {fwhm_fit:.2f} deg (target 21)")
print(f"Half-width left  of fitted peak = {left_w:.2f} deg")
print(f"Half-width right of fitted peak = {right_w:.2f} deg")
print(f"Asymmetry (right / left) = {right_w / left_w:.3f}")

## Part 3: VDF Moment Computation / VDF 모멘트 계산

측정된 카운트 $C_i$ 로부터 위상 공간 밀도 $f$ 와 모멘트 (밀도 $n$, 벌크 속도 $\vec{u}$, 온도 $T$) 를 계산한다. 핵심 공식은

$$f(\vec{v}) = \frac{C_i}{G_i \, \tau \, v_i^4 \, (\Delta E / E)},$$

그리고 모멘트는 $n = \sum_i f_i \Delta^3 v_i$, $\vec{u} = \frac{1}{n} \sum_i \vec{v}_i f_i \Delta^3 v_i$, $T = \frac{m}{3 n k_B} \sum_i |\vec{v}_i - \vec{u}|^2 f_i \Delta^3 v_i$. 좌표계는 빔 축 ($-x$) 으로부터의 polar $\theta$, azimuth $\phi$ 의 표준 spherical 좌표를 사용한다 — 이때 부피 원소는 $d^3v = v^2 \sin\theta\, dv\, d\theta\, d\phi$ 이고, log 간격 에너지의 경우 $v^2 dv = (v^3 / 2)\, d\ln E$.

From measured counts $C_i$ we recover phase-space density $f(\vec v)$, then compute moments. The crucial $1/v^4$ factor combines $J \propto v^2 f$ with the $dE \to dv$ Jacobian. Coordinates are spherical with $\theta$ measured from the $-x$ flow axis, so the volume element is $d^3 v = v^2 \sin\theta\, dv\, d\theta\, d\phi$ and $v^2 dv = (v^3 / 2)\, d\ln E$ for log-spaced energy bins.

In [ ]:
def maxwellian_3d(v_vec: np.ndarray, n: float, u_vec: np.ndarray, T: float,
                  mass: float = M_P) -> np.ndarray:
    """3-D drifting Maxwellian phase-space density.

    Args:
        v_vec: Velocity components, shape (..., 3) [m/s].
        n: Number density [m^-3].
        u_vec: Bulk-velocity vector (3,) [m/s].
        T: Temperature [K].
        mass: Particle mass [kg].

    Returns:
        Phase-space density f [s^3 / m^6].
    """
    vth_sq = 2.0 * K_B * T / mass  # (m/s)^2
    norm = n * (mass / (2.0 * np.pi * K_B * T)) ** 1.5
    delta = v_vec - u_vec
    speed_sq = np.sum(delta ** 2, axis=-1)
    return norm * np.exp(-speed_sq / vth_sq)


def speed_from_energy(energy_eV: np.ndarray, mass: float = M_P) -> np.ndarray:
    """Convert kinetic energy [eV] to non-relativistic speed [m/s].

    Args:
        energy_eV: Kinetic energy [eV].
        mass: Particle mass [kg].

    Returns:
        Speed [m/s].
    """
    return np.sqrt(2.0 * energy_eV * EV / mass)


def make_swepam_grid(n_E: int = 60, n_theta: int = 24, n_phi: int = 24,
                     E_min: float = 200.0, E_max: float = 4000.0,
                     theta_max_deg: float = 25.0):
    """Build a SWEPAM-I-like (E, theta, phi) measurement grid.

    Spherical convention: theta in [0, theta_max] is measured from the -x
    (Sunward flow) axis -- the bulk solar-wind beam center. Phi covers the
    full 2 pi azimuth around -x. Bins are uniform in theta/phi and log in E.

    Args:
        n_E: Number of energy steps (~5% spacing for default range).
        n_theta: Number of polar bins inside the cone.
        n_phi: Number of azimuth bins.
        E_min: Lowest energy [eV].
        E_max: Highest energy [eV].
        theta_max_deg: Half-cone aperture sampled [deg].

    Returns:
        Dictionary with grids and bin widths needed by ``vdf_moments``.
    """
    E_centers = np.geomspace(E_min, E_max, n_E)
    dlnE = np.log(E_centers[1] / E_centers[0])
    theta_edges = np.linspace(0.0, theta_max_deg, n_theta + 1)
    theta_centers = 0.5 * (theta_edges[:-1] + theta_edges[1:])
    dtheta_deg = theta_edges[1] - theta_edges[0]
    phi_centers = np.linspace(0.0, 360.0, n_phi, endpoint=False)
    dphi_deg = phi_centers[1] - phi_centers[0]
    EE, TT, PP = np.meshgrid(E_centers, theta_centers, phi_centers, indexing="ij")
    return {
        "E": EE, "theta": TT, "phi": PP,
        "dlnE": dlnE, "dtheta_deg": dtheta_deg, "dphi_deg": dphi_deg,
    }


def grid_to_velocity(EE: np.ndarray, TT_deg: np.ndarray, PP_deg: np.ndarray,
                     mass: float = M_P) -> np.ndarray:
    """Convert (E, theta, phi) bins to Cartesian velocity vectors.

    Spherical convention with theta from the -x axis:
        v_x = -v cos(theta)
        v_y =  v sin(theta) cos(phi)
        v_z =  v sin(theta) sin(phi)

    Args:
        EE: Energy grid [eV].
        TT_deg: Polar grid [deg].
        PP_deg: Azimuth grid [deg].
        mass: Particle mass [kg].

    Returns:
        Velocity array of shape (..., 3) [m/s].
    """
    speed = speed_from_energy(EE, mass)
    th = np.deg2rad(TT_deg)
    ph = np.deg2rad(PP_deg)
    vx = -speed * np.cos(th)
    vy = speed * np.sin(th) * np.cos(ph)
    vz = speed * np.sin(th) * np.sin(ph)
    return np.stack([vx, vy, vz], axis=-1)

In [ ]:
def vdf_moments(f: np.ndarray, v_vec: np.ndarray, grid: dict,
                mass: float = M_P) -> dict:
    """Numerical density / bulk velocity / scalar temperature from f(v).

    Volume element in (E, theta, phi):
        d^3 v = (v^3 / 2) dlnE * sin(theta) dtheta * dphi.

    Args:
        f: Phase-space density on the (E, theta, phi) grid [s^3 m^-6].
        v_vec: Velocity vectors on the grid, shape (..., 3) [m/s].
        grid: Grid dictionary from ``make_swepam_grid``.
        mass: Particle mass [kg].

    Returns:
        Dictionary with ``n`` [m^-3], ``u`` [m/s], ``T`` [K].
    """
    speed = speed_from_energy(grid["E"], mass)
    sin_theta = np.sin(np.deg2rad(grid["theta"]))
    d3v = (
        0.5 * speed ** 3 * grid["dlnE"]
        * sin_theta * np.deg2rad(grid["dtheta_deg"])
        * np.deg2rad(grid["dphi_deg"])
    )
    n = np.sum(f * d3v)
    u = np.sum(f[..., None] * v_vec * d3v[..., None], axis=(0, 1, 2)) / n
    delta = v_vec - u
    speed_sq = np.sum(delta ** 2, axis=-1)
    T = mass / (3.0 * n * K_B) * np.sum(f * speed_sq * d3v)
    return {"n": n, "u": u, "T": T}


# Build a synthetic slow-wind proton distribution and recover its moments.
grid = make_swepam_grid(n_E=80, n_theta=40, n_phi=32,
                        E_min=200.0, E_max=4000.0, theta_max_deg=25.0)
v_vec = grid_to_velocity(grid["E"], grid["theta"], grid["phi"])

n_true = 5.0e6                           # 5 cm^-3 in m^-3
u_true = np.array([-400e3, 0.0, 0.0])    # 400 km/s anti-Sunward
T_true = 5.0e4                           # 50,000 K  (about 4.3 eV)

f_grid = maxwellian_3d(v_vec, n_true, u_true, T_true, mass=M_P)
moms = vdf_moments(f_grid, v_vec, grid)

print("Truth   ->  n = {:.3f} cm^-3, |u| = {:.1f} km/s, T = {:.2e} K".format(
    n_true / 1e6, np.linalg.norm(u_true) / 1e3, T_true))
print("Moments ->  n = {:.3f} cm^-3, |u| = {:.1f} km/s, T = {:.2e} K".format(
    moms["n"] / 1e6, np.linalg.norm(moms["u"]) / 1e3, moms["T"]))

### 3.1 Counts $\to$ phase-space density round-trip / 카운트 ↔ 위상 공간 밀도 왕복

$$C_i = G_i \, \tau \, v_i^4 \, f_i \, \frac{\Delta E_i}{E_i}.$$

Geometric factor $G_i$, accumulation time $\tau$, ΔE/E의 곱이 카운트와 위상 공간 밀도 사이의 변환 인자이다. SWEPAM-I G ≈ $8 \times 10^{-6}$ cm² sr eV/eV (Table IV) 와 $\tau \approx 0.16$ s/segment를 사용해 합리적인 카운트 수준을 검증한다.

The product $G_i \tau (\Delta E / E)$ converts counts to phase-space density; using SWEPAM-I $G \approx 8 \times 10^{-6}$ cm$^2$ sr eV/eV (Table IV) and $\tau \approx 0.16$ s/segment we sanity-check the count rate.

In [ ]:
def counts_from_f(f: np.ndarray, energy_eV: np.ndarray,
                  geometric_factor_cm2_sr: float, tau_s: float,
                  dE_over_E: float, mass: float = M_P) -> np.ndarray:
    """Forward model: phase-space density f -> instrument counts.

    Uses C = G * tau * v^4 * f * (dE/E) with f in SI (s^3 m^-6).
    The geometric factor is converted from cm^2 to m^2 internally.

    Args:
        f: Phase-space density [s^3 m^-6].
        energy_eV: Energy at each grid point [eV].
        geometric_factor_cm2_sr: G in cm^2 sr eV/eV.
        tau_s: Accumulation time per pixel [s].
        dE_over_E: Relative energy bandwidth.
        mass: Particle mass [kg].

    Returns:
        Predicted counts (Poisson mean) on the same grid.
    """
    speed = speed_from_energy(energy_eV, mass)
    G_si = geometric_factor_cm2_sr * 1e-4   # cm^2 -> m^2
    return G_si * tau_s * speed ** 4 * f * dE_over_E


G_pixel = 8e-6   # SWEPAM-I CEM 4 (Table IV), cm^2 sr eV/eV
tau_pixel = 0.16  # accumulation per ESA step within a 12.8 s segment, s
dEoverE = 0.05

C_mean = counts_from_f(f_grid, grid["E"], G_pixel, tau_pixel, dEoverE)
C_obs = rng.poisson(np.clip(C_mean, 0, None))

print(f"Peak count-rate mean = {C_mean.max():.1f} counts/pixel/sample")
print(f"Total counts in synthetic VDF = {C_obs.sum():.0f}")
print(f"Pixels with mean > 1 count   = {(C_mean > 1).mean() * 100:.1f}%")

# Round-trip: invert counts -> f and re-compute moments.
speed_grid = speed_from_energy(grid["E"])
f_recovered = np.where(
    speed_grid > 0,
    C_obs / (G_pixel * 1e-4 * tau_pixel * speed_grid ** 4 * dEoverE),
    0.0,
)
moms_obs = vdf_moments(f_recovered, v_vec, grid)
print("\nRecovered from synthetic Poisson counts:")
print("  n   = {:.3f} cm^-3 (truth 5.000)".format(moms_obs["n"] / 1e6))
print("  |u| = {:.2f} km/s (truth 400.00)".format(np.linalg.norm(moms_obs["u"]) / 1e3))
print("  T   = {:.3e} K (truth 5.000e+04)".format(moms_obs["T"]))

### 3.2 1-D energy spectrum visualization / 1차원 에너지 스펙트럼 시각화

ESA voltage-step 축에 sum-over-angles 한 1차원 카운트 스펙트럼을 그린다. 좁은 supersonic 양성자 빔 ($M = u/v_{\rm th} \approx 14$)이 약 $E_0 = \frac{1}{2} m_p u^2 \approx 836$ eV 부근에 좁게 좁혀지고, $\Delta E/E \approx 14\%$의 빔 폭은 SWEPAM의 5% step에서 약 3 step에 걸쳐 분해된다.

Sum the 3-D count cube over angles to obtain the 1-D ESA-step spectrum. The supersonic ($M \approx 14$) proton beam appears as a narrow peak near $E_0 = \tfrac12 m_p u^2 \approx 836$ eV and is resolved across ~3 voltage steps.

In [ ]:
C_E = C_obs.sum(axis=(1, 2))
E_centers = grid["E"][:, 0, 0]

E_peak = 0.5 * M_P * np.linalg.norm(u_true) ** 2 / EV  # eV
vth = np.sqrt(2.0 * K_B * T_true / M_P)
mach = np.linalg.norm(u_true) / vth

fig, ax = plt.subplots(figsize=(9, 5))
ax.semilogy(E_centers, np.clip(C_E, 0.5, None), "o-", color="C0", lw=1)
ax.axvline(E_peak, color="C3", ls="--", label=f"E_peak = 0.5 m_p u\u00b2 = {E_peak:.0f} eV")
ax.set_xlabel("ESA-step center energy [eV]")
ax.set_ylabel("Counts (sum over angles)")
ax.set_title(
    f"SWEPAM-I synthetic 1-D ion spectrum (Mach = {mach:.1f}, "
    f"\u0394E/E_beam \u2248 {2 * vth / np.linalg.norm(u_true) * 100:.1f}%)"
)
ax.legend()
plt.tight_layout()
plt.show()

## Part 4: HSS vs Slow-Wind Population Separation in $n$-$T$ Space / HSS vs 저속 풍 분리

ACE/SWEPAM 관측에서 가장 강력한 통계적 발견 중 하나는 *bimodal 태양풍* (Fig. 1)이다. 코로나 홀에서 기원한 고속 풍 (HSS, $|u| > 600$ km/s)은 streamer-belt 저속 풍보다 (1) 평균 밀도가 낮고, (2) 양성자 온도가 높다. 이 셀에서는 SWEPAM 64 s 모멘트 시계열을 모사하여 두 군집이 $n$-$T$ 평면에서 깨끗이 분리됨을 보인다.

One of the most robust SWEPAM statistical results is the *bimodal solar wind* (Fig. 1). High-speed streams (HSS, $|u| > 600$ km/s) from coronal holes have lower density and higher proton temperature than streamer-belt slow wind. Here we simulate a SWEPAM 64-s moment time series and show the two populations separate cleanly in $n$-$T$ space.

In [ ]:
def synthesize_solar_wind_population(n_samples: int, mode: str,
                                     rng: np.random.Generator) -> dict:
    """Sample SWEPAM-style 64 s moments from a population.

    Reference observed ranges (rough log-normal fits to ACE 1998-2010):
        slow wind: n ~ 7-10 cm^-3, |u| ~ 350-450 km/s, T ~ (3-8)e4 K
        fast wind: n ~ 2-4 cm^-3, |u| ~ 650-800 km/s, T ~ (1.5-3)e5 K

    Args:
        n_samples: Number of synthetic 64 s samples.
        mode: ``'slow'`` or ``'fast'``.
        rng: NumPy random generator.

    Returns:
        Dict with arrays ``n_cm3``, ``u_km_s``, ``T_K``.
    """
    if mode == "slow":
        n = rng.lognormal(mean=np.log(8.0), sigma=0.25, size=n_samples)
        u = rng.normal(loc=400.0, scale=40.0, size=n_samples)
        T = rng.lognormal(mean=np.log(5.0e4), sigma=0.30, size=n_samples)
    elif mode == "fast":
        n = rng.lognormal(mean=np.log(3.0), sigma=0.25, size=n_samples)
        u = rng.normal(loc=720.0, scale=50.0, size=n_samples)
        T = rng.lognormal(mean=np.log(2.0e5), sigma=0.25, size=n_samples)
    else:
        raise ValueError("mode must be 'slow' or 'fast'")
    return {"n_cm3": n, "u_km_s": u, "T_K": T}


slow = synthesize_solar_wind_population(800, "slow", rng)
fast = synthesize_solar_wind_population(400, "fast", rng)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].scatter(slow["n_cm3"], slow["T_K"], s=10, alpha=0.55,
                color="C3", label="Slow wind (streamer belt)")
axes[0].scatter(fast["n_cm3"], fast["T_K"], s=10, alpha=0.55,
                color="C0", label="HSS (coronal hole)")
axes[0].set_xscale("log")
axes[0].set_yscale("log")
axes[0].set_xlabel("Proton density n [cm$^{-3}$]")
axes[0].set_ylabel("Proton temperature T [K]")
axes[0].set_title("SWEPAM-I bimodal population in n - T")
axes[0].legend()

speeds = np.concatenate([slow["u_km_s"], fast["u_km_s"]])
T_all = np.concatenate([slow["T_K"], fast["T_K"]])
sc = axes[1].scatter(speeds, T_all, s=10, alpha=0.55, c=speeds, cmap="plasma")
axes[1].set_xlabel("Bulk speed |u| [km/s]")
axes[1].set_ylabel("Proton temperature T [K]")
axes[1].set_yscale("log")
axes[1].set_title("T vs |u|: well-known positive correlation")
plt.colorbar(sc, ax=axes[1], label="|u| [km/s]")

plt.tight_layout()
plt.show()

In [ ]:
def classify_by_speed(u_km_s: np.ndarray, threshold: float = 550.0) -> np.ndarray:
    """Threshold-based slow vs HSS classifier.

    Args:
        u_km_s: Bulk speed [km/s].
        threshold: Decision boundary [km/s].

    Returns:
        Array of labels: 0 = slow, 1 = HSS.
    """
    return (u_km_s >= threshold).astype(int)


u_all = np.concatenate([slow["u_km_s"], fast["u_km_s"]])
n_all = np.concatenate([slow["n_cm3"], fast["n_cm3"]])
T_all_arr = np.concatenate([slow["T_K"], fast["T_K"]])
y_true = np.concatenate([np.zeros(len(slow["u_km_s"])), np.ones(len(fast["u_km_s"]))])
y_pred = classify_by_speed(u_all)

TP = int(((y_pred == 1) & (y_true == 1)).sum())
TN = int(((y_pred == 0) & (y_true == 0)).sum())
FP = int(((y_pred == 1) & (y_true == 0)).sum())
FN = int(((y_pred == 0) & (y_true == 1)).sum())
accuracy = (TP + TN) / len(y_true)

print("Speed-threshold (|u| >= 550 km/s) classification:")
print(f"  TP = {TP}, TN = {TN}, FP = {FP}, FN = {FN}")
print(f"  Accuracy = {accuracy:.3f}")

for label, mask, name in [(0, y_true == 0, "slow"), (1, y_true == 1, "fast/HSS")]:
    n_med = np.median(n_all[mask])
    T_med = np.median(T_all_arr[mask])
    u_med = np.median(u_all[mask])
    print(f"  {name:8s} median  n = {n_med:5.2f} cm^-3,  |u| = {u_med:5.0f} km/s,  T = {T_med:.2e} K")

## Summary / 요약

| Concept / 개념 | This Paper / 이 논문 | Modern Equivalent / 현대 동등물 |
|---|---|---|
| Spherical-section ESA + CEM | SWEPAM-I (105°, 16 CEM), SWEPAM-E (120°, 7 CEM) | Solar Orbiter SWA (PAS / EAS / HIS); PSP SWEAP/SPAN |
| Separable 3-D response $T(E, \theta, \phi)$ | Gosling et al. 1978 decomposition | Same form used in SWA, FPI (MMS), SPAN |
| 64 s 3-D VDF + moments at L1 | DSWI 96 px, RTSW 33 px | DSCOVR PlasMag, IMAP/SWAPI |
| Bimodal solar wind in $n$-$T$ | Fig. 1 (Ulysses); ACE 24/7 verification | Active classifier inputs in modern HSS catalogs |
| Real-time L1 monitor for forecasting | RTSW $\to$ NOAA SEC, 30-60 min lead time | DSCOVR (2015), IMAP (2025), Space Weather Follow-On |

### Notebook check-list / 노트북 체크리스트
- 분석기 상수 $K = r_0 / (2d)$ 검증 (SWEPAM-I 17.6 vs 17.1, SWEPAM-E 5.99 vs 4.3) / Analyzer-constant verification.
- 분리 가능 응답 함수 (Gaussian + 사다리꼴) 시각화 / Separable response visualization.
- SWEPAM-E 7-CEM 폴라 fan과 왜곡 가우시안 / SWEPAM-E 7-channel polar fan with skewed Gaussian.
- 합성 카운트로부터 $n$, $\vec{u}$, $T$ 모멘트 회복 / VDF moment round-trip from synthetic Poisson counts.
- HSS vs 저속 풍 $n$-$T$ 분리 (속도 임계값 분류기) / Bimodal classification.

### 한계 / Limitations
이 노트북은 ESA 응답을 *분리 가능*하다고 가정했으나 실제로는 (Fig. 13) energy-azimuth 결합이 있다. 또한 합성 모멘트는 단일 Maxwellian만 가정 — 실제 SWEPAM은 양성자/알파 분리, kappa 분포, halo 전자 등 비-Maxwellian 효과를 다루어야 한다.

This notebook assumes a strictly separable ESA response, while in reality (Fig. 13) energy and azimuth are coupled. The moment recovery uses a single Maxwellian, whereas real SWEPAM analysis must separate protons from alphas and handle non-Maxwellian features (kappa distributions, halo electrons).